[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/saltfingers/saltfingers.ipynb)

# Salt Fingers

This example computes salt fingers, the double-diffusive convection of warm salty water lying over cold fresh water across a triangulated periodic tank. Warmth makes the upper layer light while dissolved salt makes it heavy, and with enough warmth on top the overall density profile is stably stratified, lighter water over heavier. Even so, heat conducts through seawater roughly a hundred times faster than salt diffuses: a salty parcel nudged downward loses its warmth sideways to its cooler neighbours long before it loses its salt, becomes heavier than they are and keeps sinking, while a fresh parcel nudged upward warms and keeps rising. From slight salinity noise along the interface, narrow alternating fingers grow whenever the stabilizing thermal-to-haline density ratio stays below the diffusivity ratio of heat to salt, following [Radko's Woods Hole GFD lectures (2008)](https://gfd.whoi.edu/wp-content/uploads/sites/18/2018/03/lecture10_136325.pdf).

Joining the projective plane with four degenerate directions for volume, mass, entropy and salt places vertices by projective weight while modelling each fluid parcel as an intrinsic antivector `volume + (density - expansion * warmth + contraction * saltiness) * mass + warmth * entropy + saltiness * salt`. Thermodynamic drives take the form `-gradients ^ s` and `-gradients ^ n`, mapping to entropy and salt fluxes at their distinct conductivities, while gravity's drive `-gravity * ym` pulls on mass alone because its commutator with `volume`, `entropy` and `salt` vanishes. Advection transports whole parcels in one flux `velocity & parcels`, and pairing the dual-cell outflow with `s` and `n` reads out both advected scalars without separate transport equations.

In vector-calculus notation, the creeping Oberbeck–Boussinesq equations split momentum, incompressibility, heat and salinity across coupled partial differential equations, $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g(\alpha T - \beta S)\,\hat{\mathbf y}$, $\nabla\cdot\mathbf v = 0$, $\partial_t T + \mathbf v\cdot\nabla T = \kappa_T\nabla^2 T$ and $\partial_t S + \mathbf v\cdot\nabla S = \kappa_S\nabla^2 S$, or eliminate pressure via a scalar stream function $\mathbf v = (\partial_y\psi, -\partial_x\psi)$ and scalar vorticity $\omega = -\nabla^2\psi$ satisfying $\partial_t\nabla^2\psi = \nu\nabla^4\psi - g(\alpha\,\partial_x T - \beta\,\partial_x S)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Callable, Iterator
from itertools import islice

from numga import Algebra, NumpyContext, stack
from numga.sparse import Sparse, SparseExtensor, spdiag
from examples.fluids.saltfingers import render, scenarios
from examples.fluids.saltfingers.plumbing import tank
from examples.util.animation import show_states

# The projective plane for spatial placement, joined with degenerate directions for volume, mass, entropy and salt.
space = Algebra("x+y+w0")
quantities = Algebra("v0m0s0n0")
ga = space * quantities
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Antiscalar = ga.gatype.antiscalar()                                           # Antiscalar-
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# Spatial coordinate planes and drives coupling spatial gradients to material quantities.
Upright = Plane.derive.within("x y")                                          # Vector+
Drive = ga.gatype.bivector().derive.touching("x y", quantities)               # Bivector+
Flux = ga.gatype.antibivector()                                               # Antibivector-
# Horizontal and vertical spatial directions and their conjugate coordinate planes.
length = mv.x.dual()                                                          # [] Point-
height = mv.y.dual()                                                          # [] Point-
directions = stack((length, height))                                          # [2] Point-
readings = stack((mv.x, mv.y))                                                # [2] Plane+
# Reference origin at unit projective weight, unit material quantities, and their joint complement for spatial areas.
origin = mv.w.dual()                                                          # [] Point-
volume = mv.v.dual()                                                          # [] Point-
mass = mv.m.dual()                                                            # [] Point-
entropy = mv.s.dual()                                                         # [] Point-
salt = mv.n.dual()                                                            # [] Point-
unread = volume & mass & entropy & salt                                       # [] Trivector-

## Stratified tank setup

A triangulated strip one depth high closes periodically along its length so the tank has a floor and a ceiling without side walls, while an opened copy repeats the seam column one period further along so triangle edges have continuous coordinates. Warm salty water rests above cold fresh water across a thin mid-depth interface seeded with small salinity perturbations, with lengths in tank depths, times in thermal diffusion times across the depth, and temperature and salinity in their top-to-bottom differences.

In [ ]:
# Closed periodic and seam-opened tank topologies, vertex readings between them, and projective vertex positions.
closed, opened, closing, opening, places = tank(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (mv(Upright, places) + mv.w).dual().field()                        # Point-[V_opened]
temperature, salinity = scenarios.layer_profiles(closing * (mv.y & vertices), scenarios.disturbance)  # Scalar+[V], Scalar+[V]
render.draw_tank(opened, vertices, opening, scenarios.period, temperature, salinity);

The upper panel shows temperature, red where warm and blue where cold, and the lower panel shows salinity, brown where salty and green where fresh, across the perturbed mid-depth interface.

## Double-diffusive fingering

A vertex stream function represents the incompressible flow, where a unit rise at a triangle corner drives fluid parallel to the opposite edge. Pairing gravity's drive with the parcel-carrying corner flows `pull & (flows & parcels)` supplies both thermal and haline buoyancy from one expression because warmth lowers parcel mass while salt raises it. At a density ratio of two, thermal buoyancy initially outweighs haline buoyancy by a factor of two, while heat conducts ten times faster than salt diffuses so the triangulation resolves the sharp salt boundaries. Stepping the tank takes about fifteen seconds, and the animation begins just before the fingers emerge.

In [ ]:
# Fluid parameters, thermal Rayleigh number and density ratio, and time-stepping counts.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
contraction = expansion / scenarios.ratio
salt_diffusivity = scenarios.salt_diffusivity
gravity = scenarios.rayleigh * viscosity / expansion
dt, n_steps, n_stride, n_flat = scenarios.dt, scenarios.n_steps, scenarios.n_stride, scenarios.n_flat

In [ ]:
# Triangle geometry on the opened strip, pulled back to the closed periodic vertex topology.
opened_corners = opened.incidence[2, 0] * vertices                            # [3] Point-[F]
facing = opened.off[2, 0] * vertices                                          # [2, 3] Point-[F]
sides = facing[0] & facing[1]                                                 # [3] Flux-[F]
# Twice each triangle's signed area in the projective plane, pairing past the four material directions.
double_areas = opened_corners[0] & opened_corners[1] & opened_corners[2] & unread  # Antiscalar-[F]
areas = double_areas.norm() / 2                                               # Scalar+[F]
# Pure spatial simplex gradients: each plane vanishes along the facing side and rises to one at its corner.
gradients = ((sides & unread) / (sides & unread & opened_corners)).cast(Upright)  # [3] Plane+[F]
corners = closed.incidence[2, 0]                                              # [3] Scalar+[F <- V]
dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)                   # Scalar+[V]
corner_mean = corners.mean(axis=0)                                            # Scalar+[F <- V]
# Unit stream function at a corner drives flow along the oriented edge facing it.
edges = opened.boundary[2, 1] * (opened.d[0] * vertices)                      # [3] Point-[F]
flows = edges / 2                                                             # [3] Point-[F]
velocities = flows / areas                                                    # [3] Point-[F]
# Triangle velocity from vertex stream values, and dual-cell outflow integrated against each corner's simplex gradient.
stream_velocity = (spdiag(velocities * Scalar) * corners).sum(axis=0)         # (Point- <- Scalar+)[F <- V]
cell_outflow = (corners.adjugate() * spdiag(areas * (gradients ^ Flux))).sum(axis=0)  # (Point- <- Flux-)[V <- F]

In [ ]:
# Kinetic energy and viscous vorticity dissipation forms on the vertex stream function.
inertia = density * (readings * (readings & Point)).sum(axis=0)               # [] Plane+ <- Point-
local = areas * (inertia(velocities[:, None]) & velocities[None, :])          # [3, 3] Scalar+[F]
kinetic = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
vorticity = spdiag(1 / (density * dual_areas)) * kinetic                      # Scalar+[V <- V]
viscous = (viscosity / density) * kinetic * vorticity                         # Scalar+[V <- V]
# Thermal and haline drives on each corner's simplex gradient, mapped to entropy and salt fluxes.
heat_drives, salt_drives = -(gradients ^ mv.s), -(gradients ^ mv.n)           # [3] Drive+[F], [3] Drive+[F]
conductivity = ((directions & entropy) * ((directions & entropy) & Drive)).sum(axis=0)  # [] Flux- <- Drive+
diffusivity = salt_diffusivity * ((directions & salt) * ((directions & salt) & Drive)).sum(axis=0)  # [] Flux- <- Drive+
local = areas * (mv.s & (gradients[:, None] ^ conductivity(heat_drives[None, :])))  # [3, 3] Scalar+[F]
conduction = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
local = areas * (mv.n & (gradients[:, None] ^ diffusivity(salt_drives[None, :])))  # [3, 3] Scalar+[F]
diffusion = (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)  # Scalar+[V <- V]
pull = -(gravity * mv.ym)                                                     # [] Drive+

In [ ]:
def wall_solver(form: SparseExtensor, interior: Sparse) -> Callable[[Scalar, Scalar], Scalar]:
    """Linear solver for a vertex operator with Dirichlet values held on the boundary vertices."""
    free = (interior * form * interior.adjugate()) * Scalar                   # (Scalar+ <- Scalar+)[V_free <- V_free]
    return lambda rhs, walls: walls + interior.adjugate() * free.solve(interior * (rhs - form * walls))


# Pre-factored implicit solvers and fixed floor-and-ceiling boundary values.
interior = closed.reading(closed.interior[0], 0)                              # Scalar+[V_free <- V]
flow_step = wall_solver(kinetic / dt + viscous, interior)
heat_step = wall_solver(dual_areas / dt + conduction, interior)
salt_step = wall_solver(dual_areas / dt + diffusion, interior)
wall_temperature = temperature - interior.adjugate() * (interior * temperature)  # Scalar+[V]
wall_salinity = salinity - interior.adjugate() * (interior * salinity)        # Scalar+[V]

In [ ]:
def fingering(temperature: Scalar, salinity: Scalar) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """Yield the stream function, temperature and salinity fields every `n_stride` steps as salt fingers grow."""
    stream = 0 * temperature                                                  # Scalar+[V]
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature, salinity
        if step == n_steps:
            break
        # Boussinesq parcel composition per unit volume: thermal expansion lowers mass while salt contraction raises it.
        warmth = corner_mean * temperature                                    # Scalar+[F]
        saltiness = corner_mean * salinity                                    # Scalar+[F]
        parcels = (
            volume
            + (density - expansion * warmth + contraction * saltiness) * mass
            + warmth * entropy
            + saltiness * salt
        )                                                                     # Point-[F]
        # Buoyancy power of gravity's drive on each corner's parcel-carrying flow.
        load = (corners.adjugate() * (pull & (flows & parcels[..., None]))).sum(axis=-1)  # Scalar+[V]
        stream = flow_step(kinetic / dt * stream + load, 0 * stream)
        # Whole-parcel advection read out in entropy and salt, followed by implicit conduction and salt diffusion.
        velocity = stream_velocity(stream)                                    # Point-[F]
        flux = velocity & parcels                                             # Flux-[F]
        outflow = cell_outflow(flux)                                          # Point-[V]
        temperature = heat_step(dual_areas / dt * temperature - (mv.s & outflow), wall_temperature)
        salinity = salt_step(dual_areas / dt * salinity - (mv.n & outflow), wall_salinity)


states = show_states(
    islice(fingering(temperature, salinity), n_flat, None),
    render.animate,
    opened,
    vertices,
    opening,
    scenarios.period,
    name="saltfingers",
    duration_ms=scenarios.frame_duration_ms,
    n_frames=n_steps // n_stride + 1 - n_flat,
    colors=scenarios.n_animation_colours,
)

Small interface ripples organize into roughly ten alternating plumes of sinking salty water and rising fresh water across the tank, each about one thermal diffusion scale wide. In the upper temperature panel the plumes look diffuse because rapid sideways conduction equilibrates their warmth with neighbouring columns, whereas in the lower salinity panel they keep sharp boundaries and mushroom-like vortex caps that sustain the buoyancy drive.

In [ ]:
# checks
# Gravity pulls parcel mass downward while leaving volume, entropy and salt untouched.
for part, expected in ((mass, -1.0), (volume, 0.0), (entropy, 0.0), (salt, 0.0)):
    assert (((mv.y & (-mv.ym).commutator(part)) - expected).abs() < 1e-12).all()
# Across the mid-depth plane, salinity retains nearly the full layer contrast and varies over twice as strongly as temperature.
middle = ((mv.y & vertices.batch()) - 0.5).abs() < 1e-12
_, last_temperature, last_salinity = states[-1]
salinities = (opening * last_salinity).batch()[middle]
temperatures = (opening * last_temperature).batch()[middle]
assert bool(salinities[salinities.argmax()] - salinities[salinities.argmin()] > 0.9)
assert bool((temperatures - temperatures.mean()).squared().mean() < (salinities - salinities.mean()).squared().mean() / 4)